In [1]:
# Cell 1: Imports and Setup

import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm


print("✅ All libraries imported successfully!")


2025-06-14 23:44:43.306817: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749959083.369339  390293 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749959083.387676  390293 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749959083.513569  390293 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749959083.513586  390293 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749959083.513588  390293 computation_placer.cc:177] computation placer alr

✅ All libraries imported successfully!


In [2]:
import sys
sys.path.append("/hpcstor6/scratch01/a/a.kanamarlapudi001/BreastCancerMultimodel")  # Adjust if needed

from fusion_model import FusionModel


In [3]:
# Cell 2: Set dataset path and collect .npz files

import glob

# Folder containing Cancer and NonCancer subfolders of .npz files
NPZ_DIR = "/hpcstor6/scratch01/a/a.kanamarlapudi001/balanced_dataset_from_2dresized/npz"

cancer_npz = sorted(glob.glob(os.path.join(NPZ_DIR, "Cancer", "*.npz")))
noncancer_npz = sorted(glob.glob(os.path.join(NPZ_DIR, "NonCancer", "*.npz")))

all_npz_files = cancer_npz + noncancer_npz
labels = [1] * len(cancer_npz) + [0] * len(noncancer_npz)

print(f"Total NPZ files: {len(all_npz_files)}")
print(f"Cancer: {len(cancer_npz)}, NonCancer: {len(noncancer_npz)}")


Total NPZ files: 14702
Cancer: 7351, NonCancer: 7351


In [4]:
# Cell 3: Load data from NPZ files
X_images = []
X_metadata = []
y = []

for path, label in tqdm(zip(all_npz_files, labels), total=len(all_npz_files), desc="Loading NPZ files"):
    data = np.load(path)
    X_images.append(data["image"])          # Shape: (224, 224)
    X_metadata.append(data["metadata"])     # Shape: (2,)
    y.append(label)

# Convert to numpy arrays
X_images = np.array(X_images)
X_metadata = np.array(X_metadata)
y = np.array(y)

# Normalize image data and expand dims to (224, 224, 3)
X_images = X_images.astype(np.float32) / 255.0
X_images = np.expand_dims(X_images, axis=-1)  # (N, 224, 224, 1)
X_images = np.repeat(X_images, 3, axis=-1)    # (N, 224, 224, 3)

# One-hot encode labels
y = to_categorical(y, num_classes=2)

print(f"X_images: {X_images.shape}, X_metadata: {X_metadata.shape}, y: {y.shape}")


Loading NPZ files: 100%|█████████████████████████████████████████████████████████████████| 14702/14702 [00:14<00:00, 986.08it/s]


X_images: (14702, 224, 224, 3), X_metadata: (14702, 2), y: (14702, 2)


In [5]:
# Cell 4: Train/Val/Test Split
from sklearn.model_selection import train_test_split

# First split: Train vs Temp (Val+Test)
X_train_img, X_temp_img, X_train_meta, X_temp_meta, y_train, y_temp = train_test_split(
    X_images, X_metadata, y, test_size=0.4, stratify=y, random_state=42
)

# Second split: Validation vs Test
X_val_img, X_test_img, X_val_meta, X_test_meta, y_val, y_test = train_test_split(
    X_temp_img, X_temp_meta, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("✅ Dataset split complete!")
print(f"Train size: {X_train_img.shape[0]}")
print(f"Val size:   {X_val_img.shape[0]}")
print(f"Test size:  {X_test_img.shape[0]}")


✅ Dataset split complete!
Train size: 8821
Val size:   2940
Test size:  2941


In [6]:
# Cell 5: Initialize, Compile, and Train Model

# One-hot encode labels
y_train_cat = to_categorical(y_train, num_classes=2)
y_val_cat = to_categorical(y_val, num_classes=2)

# Create weights folder
os.makedirs("weights", exist_ok=True)

# Initialize model
fusion = FusionModel(input_shape=(224, 224, 3), metadata_dim=2, num_classes=2, weights_dir="weights")
fusion.compile_model()

# Train the model
history = fusion.train(
    X_train_img, X_train_meta, y_train_cat,
    X_val_img, X_val_meta, y_val_cat
)


I0000 00:00:1749959124.067134  390293 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38380 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:07:00.0, compute capability: 8.0


ValueError: When using `save_weights_only=True` in `ModelCheckpoint`, the filepath provided must end in `.weights.h5` (Keras weights format). Received: filepath=weights/best_weights.h5

In [7]:
import sys
import importlib

# Remove from sys.modules to clear old cached version
sys.modules.pop("fusion_model", None)

# Now import fresh version
import fusion_model
importlib.reload(fusion_model)

# Now use the class
FusionModel = fusion_model.FusionModel


In [8]:
fusion = FusionModel(input_shape=(224, 224, 3), metadata_dim=2, num_classes=2)
fusion.compile_model()
history = fusion.train(X_train_img, X_train_meta, y_train_cat, X_val_img, X_val_meta, y_val_cat)


ValueError: When using `save_weights_only=True` in `ModelCheckpoint`, the filepath provided must end in `.weights.h5` (Keras weights format). Received: filepath=weights/best_weights.h5